# Testing models

In [1]:
import torch
import torch.nn as nn
from models.blocks import ConvNeXtcausal
from torch.nn.utils.parametrizations import weight_norm
from transformers import EncodecModel
import sys
sys.path.append('../stable-audio-3')
from stable_audio_3 import AutoencoderModel
from utils.mel import MelSpectra

/home/lois/miniconda3/envs/latency/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention


In [38]:
mel_extractor = MelSpectra(
    sample_rate=24000,
    n_fft=1024,
    hop_length=256,
    n_mels=128
)

In [ ]:
class EncoderFast(nn.Module):
    def __init__(self, in_channels: int, dim: int, latent_dim: int, 
                 inter_channels: int, num_blocks: int):
        super(EncoderFast, self).__init__()

        strides = [8, 8, 8] 

        self.pad_input = nn.ConstantPad1d((6, 0), 0)
        self.conv1 = weight_norm(nn.Conv1d(in_channels, dim, kernel_size=7, padding=0))
        self.stages = nn.ModuleList()

        for s in strides:
            blocks = [ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)]
            stage = nn.Sequential(
                *blocks,
                nn.ConstantPad1d((s - 1, 0), 0),  # causal padding
                nn.Conv1d(dim, dim, kernel_size=s, stride=s, padding=0)
            )
            self.stages.append(stage)

        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.proj = nn.Linear(dim, latent_dim)

    def forward(self, x):
        x = self.pad_input(x)
        x = self.conv1(x)                # (B, dim, T)

        for stage in self.stages:
            x = stage(x)                 # (B, dim, T/stride_total)

        x = x.transpose(1, 2)           # (B, T', dim)
        x = self.norm(x)
        x = self.proj(x)                 # (B, T', latent_dim)
        x = x.transpose(1, 2)           # (B, latent_dim, T')
        return x

In [50]:
audio = torch.randn(1, 1, 573)
encoder = EncoderFast(1, 128, 32, 256, 3)
emb = encoder(audio)
print(emb.shape)

torch.Size([1, 32, 2])


In [25]:
x = torch.ones(1, 1, 24000)
model = EncodecModel.from_pretrained('facebook/encodec_24khz')
with torch.no_grad():
    emb = model.encoder(x)
    #emb = (emb - emb.mean(dim=-1, keepdim=True)) / (emb.std(dim=-1, keepdim=True) + 1e-5)
print(f"Encoder output shape: {emb.shape}")
print(f"Encoder output mean: {emb.mean().item():.4f}, std: {emb.std().item():.4f}")

Loading weights: 100%|██████████| 252/252 [00:00<00:00, 3322.05it/s]


Encoder output shape: torch.Size([1, 128, 75])
Encoder output mean: -1.1315, std: 8.3578


In [3]:
sample_rate = 24000
x = torch.randn(1, 1, 12000)
ae = AutoencoderModel.from_pretrained("same-s")
with torch.no_grad():
    emb = ae.encode(x, sample_rate)
print(f"Autoencoder output shape: {emb.shape}")

Autoencoder output shape: torch.Size([1, 256, 6])


In [3]:
class Decoder(nn.Module):
    def __init__(self, in_channels: int, dim: int, shift_dim: int, inter_channels: int, num_blocks: int):
        super(Decoder, self).__init__()
        self.pad_input = nn.ConstantPad1d((6, 0), 0)
        self.conv = nn.Conv1d(in_channels, dim, kernel_size=7, padding=0)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.blocks = nn.ModuleList([ConvNeXtcausal(dim, inter_channels) for _ in range(num_blocks)])
        self.linear1 = nn.Linear(dim, dim)
        self.linear2 = nn.Linear(dim, shift_dim, bias=False) 
        # (B, shift_dim, T) -> (B, 1 , shift_dim * T)
    
    def forward(self, x):
        x = self.pad_input(x)
        x = self.conv(x)
        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.norm(x)
        x = x.transpose(1, 2)  # (B, dim, T)

        for block in self.blocks:
            x = block(x)

        x = x.transpose(1, 2)  # (B, T, dim)
        x = self.linear1(x)
        x = self.linear2(x) # (B, T, shift_dim)
        x = x.view(x.size(0), -1) # (B, shift_dim * T)

        return x

In [21]:
s_dim = x.size(-1) // emb.size(-1)
decoder = Decoder(in_channels=128, dim=512, shift_dim=s_dim, inter_channels=256, num_blocks=2)
decoder = decoder.to(emb.device) 
y = decoder(emb)
print(f"Decoder output shape: {y.shape}")

Decoder output shape: torch.Size([1, 24000])


In [26]:
with torch.no_grad():
    y = model.decoder(emb)

In [27]:
import torch.nn.functional as F
mel_original = mel_extractor(x)
mel_reconstructed = mel_extractor(y)
print(f"EnCodec native mel loss: {F.l1_loss(mel_reconstructed, mel_original).item():.4f}")

EnCodec native mel loss: 53.4766


In [ ]:
from encodec import EncodecModel
from encodec.utils import convert_audio
import torchaudio

wav = torch.load('/home/lois/wavenext/logs/05-06_at_03_27_42/wavenext/version_0/audio_epoch_80/sample_0_real.wav')

model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(6.0)

with torch.no_grad():
    encoded_frames = model.encode(wav)  # liste de (codes, scale)
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)  # [B, n_q, T]
    emb = model.quantizer.decode(codes.transpose(0, 1))
    print(f"emb shape : {emb.shape}")  # [1, 128, 75]

/home/lois/miniconda3/envs/latency/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


emb shape : torch.Size([2, 128, 1])
emb shape : torch.Size([2, 128, 1])
torch.Size([2, 128, 2])


torch.Size([2, 2, 128])

In [5]:
import torch
import yaml
from models.wavenext_prior import WaveNeXtLatent
import torchaudio

def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

config = load_config('/home/lois/wavenext/config_prior_24k.yaml')

model = WaveNeXtLatent(
        dim=config['dim'],
        sample_rate=config['sample_rate'],
        fft_dim=config['fft_dim'],
        shift_dim=config['shift_dim'],
        n_mels=config['n_mels'],
        k=config['k'],
        lr_g=config['learning_rate_g'],
        lr_d=config['learning_rate_d'],
        prior=config['prior']
    ).to('cuda')

model.load_state_dict(torch.load("/home/lois/wavenext/checkpoints/05-06_at_03_27_42/wavenext-epoch=88-val_mel_loss=0.893.ckpt")['state_dict'])
model.eval()

/home/lois/miniconda3/envs/latency/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


WaveNeXtLatent(
  (decoder): Decoder(
    (pad_input): ConstantPad1d(padding=(6, 0), value=0)
    (conv): Conv1d(128, 512, kernel_size=(7,), stride=(1,))
    (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
    (blocks): ModuleList(
      (0-7): 8 x ConvNeXtcausal(
        (pad): ConstantPad1d(padding=(6, 0), value=0)
        (depthwise): Conv1d(512, 512, kernel_size=(7,), stride=(1,), groups=512)
        (norm): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        (pointwise1): Linear(in_features=512, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (pointwise2): Linear(in_features=1536, out_features=512, bias=True)
      )
    )
    (linear1): Linear(in_features=512, out_features=512, bias=True)
    (linear2): Linear(in_features=512, out_features=320, bias=False)
  )
  (discriminator_mpd): MPD(
    (discriminators): ModuleList(
      (0-4): 5 x OnePeriod(
        (conv): ModuleList(
          (0): ParametrizedConv2d(
            1, 3

In [ ]:
wav = torchaudio.load('/home/lois/wavenext/logs/05-06_at_03_27_42/wavenext/version_0/audio_epoch_80/sample_0_real.wav')[0].unsqueeze(0).to(model.device)
with torch.no_grad():
    encoded_frames = model.encoder.encode(wav)
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)
    emb = model.encoder.quantizer.decode(codes.transpose(0, 1))
    fake = model.decoder(emb.to(model.device))

from IPython.display import Audio
Audio(fake.cpu().squeeze(), rate=24000)

torch.Size([1, 1, 24000])


AttributeError: 'EncodecEncoder' object has no attribute 'encode'

In [8]:
from encodec import EncodecModel

model = EncodecModel.encodec_model_24khz().to('cuda')
model.set_target_bandwidth(6.0)

with torch.no_grad():
    encoded_frames = model.encode(wav) 
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)  # [B, n_q, T]
    emb = model.quantizer.decode(codes.transpose(0, 1))
    fake = model.decoder(emb)

Audio(fake.cpu().squeeze(), rate=24000)

In [35]:
from encodec import EncodecModel

wav = torch.randn(1, 2, 48000).to('cuda')

model = EncodecModel.encodec_model_48khz().to('cuda')
model.set_target_bandwidth(6.0)

with torch.no_grad():
    encoded_frames = model.encode(wav) 
    codes = torch.cat([f[0] for f in encoded_frames], dim=-1)  # [B, n_q, T]
    emb = model.quantizer.decode(codes.transpose(0, 1))

print(f"Encoder output shape: {emb.shape}")


Encoder output shape: torch.Size([1, 128, 152])
